# OpenClaw-Colab: LLM Training Agent

An AI agent based on **OpenClaw architecture principles** that:
- **Trains LLMs** via Unsloth/QLoRA fine-tuning
- **Auto-fixes errors** with rule-based + LLM-powered recovery
- **Discovers datasets** automatically from Hugging Face
- **Uses LLM as the brain** for reasoning and planning

> Architecture: Gateway (orchestrator) → Sessions (isolation) → Tools (skills) → Memory (files) → Heartbeat (monitoring) → Brain (LLM)

---
## Architecture Overview

| Layer | OpenClaw | Our Agent |
|---|---|---|
| **Control Plane** | Gateway (WS server) | `Gateway` - orchestrator |
| **Isolation** | Per-session lane queues | `SessionManager` - per-run isolation |
| **Memory** | Markdown/YAML files | `MemoryStore` - JSON/Markdown files |
| **Extensibility** | SKILL.md files | `Tool Registry` - skills system |
| **Proactivity** | HEARTBEAT.md loop | `HeartbeatMonitor` - training monitor |
| **Reasoning** | Model-agnostic BYOK | `LLMBrain` - LLM-driven decisions |
| **Self-healing** | Tool execution | `AutoFixerTool` - error recovery |

*(Inspired by [github.com/openclaw/openclaw](https://github.com/openclaw/openclaw))*

In [ ]:
# ============================================================
# CELL 1: Install Dependencies (Run once)
# ============================================================
import sys, subprocess, importlib

packages = [
    'transformers>=4.36.0', 'datasets>=2.14.0', 'accelerate>=0.24.0',
    'peft>=0.6.0', 'trl>=0.7.0', 'bitsandbytes>=0.41.0',
    'scipy', 'sentencepiece', 'huggingface_hub',
    'openai',  # for LLM brain
]

for pkg in packages:
    name = pkg.split('>=')[0].split('==')[0].replace('-', '_')
    try:
        importlib.import_module(name)
        print(f'  \u2713 {pkg}')
    except ImportError:
        print(f'  Installing {pkg}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg], timeout=120)

# Optional: Unsloth for fast training on T4
try:
    import unsloth
    print('  \u2713 unsloth (fast training available)')
except ImportError:
    print('  Installing unsloth...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'unsloth'], timeout=180)

print('\nAll dependencies ready!')

In [ ]:
# ============================================================
# CELL 2: Clone/load the agent code
# ============================================================
import os, sys
from pathlib import Path

# Mount Google Drive for persistence (optional)
from google.colab import drive
drive.mount('/content/drive')

# The agent code is self-contained in this notebook.
# All classes are defined below. Run this cell to load everything.
print('\u2713 Drive mounted at /content/drive')
print('\u2713 Ready to initialize the agent')

In [ ]:
# ============================================================
# CELL 3: Import the agent code
# ============================================================
import sys, os, json, time, logging, threading, re, uuid
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Optional, Callable, Any
from dataclasses import dataclass, field, asdict
from enum import Enum
from collections import deque

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(name)s] %(levelname)s: %(message)s')
logger = logging.getLogger('openclaw')

print('\u2713 Imports loaded')

---
## Core Architecture Layer

Building the OpenClaw-inspired control plane: Gateway, Sessions, Memory, Heartbeat, and Brain.

In [ ]:
# ============================================================
# CELL 4: Configuration
# ============================================================
@dataclass
class AgentConfig:
    workspace_root: Path = Path('/content/openclaw_workspace')
    memory_dir: Path = Path('/content/openclaw_workspace/memory')
    skills_dir: Path = Path('/content/openclaw_workspace/skills')
    logs_dir: Path = Path('/content/openclaw_workspace/logs')
    models_dir: Path = Path('/content/openclaw_workspace/models')
    datasets_dir: Path = Path('/content/openclaw_workspace/datasets')
    huggingface_token: Optional[str] = None
    openai_api_key: Optional[str] = None
    anthropic_api_key: Optional[str] = None
    llm_provider: str = 'openai'
    llm_model: str = 'gpt-4o-mini'
    llm_temperature: float = 0.3
    heartbeat_interval_seconds: int = 60
    max_retries_on_error: int = 3
    max_concurrent_sessions: int = 2
    default_training_config: dict = field(default_factory=lambda: {
        'model_name': 'unsloth/mistral-7b-bnb-4bit',
        'max_seq_length': 2048,
        'batch_size': 2,
        'gradient_accumulation_steps': 4,
        'learning_rate': 2e-4,
        'num_train_epochs': 3,
        'lora_r': 16,
        'lora_alpha': 16,
        'lora_dropout': 0.0,
        'optim': 'adamw_8bit',
        'warmup_steps': 10,
    })

config = AgentConfig()

# Create workspace directories
for d in [config.memory_dir, config.skills_dir, config.logs_dir,
          config.models_dir, config.datasets_dir]:
    d.mkdir(parents=True, exist_ok=True)

# Set API keys from environment or Colab secrets
config.openai_api_key = config.openai_api_key or os.environ.get('OPENAI_API_KEY') or ''
config.huggingface_token = config.huggingface_token or os.environ.get('HF_TOKEN') or ''

print(f'\u2713 Workspace: {config.workspace_root}')
print(f'\u2713 LLM Brain: {config.llm_provider}/{config.llm_model}')

In [ ]:
# ============================================================
# CELL 5: Memory Store (OpenClaw file-based memory)
# ============================================================
class MemoryStore:
    def __init__(self, memory_dir):
        self.memory_dir = Path(memory_dir)
        self._experiments_dir = self.memory_dir / 'experiments'
        self._errors_dir = self.memory_dir / 'errors'
        self._datasets_dir = self.memory_dir / 'datasets'
        self._agent_dir = self.memory_dir / 'agent'
        self._heartbeat_dir = self.memory_dir / 'heartbeat'
        for d in [self._experiments_dir, self._errors_dir,
                  self._datasets_dir, self._agent_dir, self._heartbeat_dir]:
            d.mkdir(exist_ok=True)

    def create_experiment(self, name, config=None):
        exp_id = f'{name}_{datetime.now().strftime("%Y%m%d_%H%M%S")}'
        record = {'id': exp_id, 'name': name, 'config': config or {},
                  'status': 'created', 'created_at': datetime.now().isoformat(),
                  'updated_at': datetime.now().isoformat(), 'metrics': {},
                  'artifacts': [], 'errors': []}
        (self._experiments_dir / f'{exp_id}.json').write_text(
            json.dumps(record, indent=2, default=str))
        return exp_id

    def update_experiment(self, exp_id, **updates):
        path = self._experiments_dir / f'{exp_id}.json'
        if not path.exists(): return
        record = json.loads(path.read_text())
        for k, v in updates.items():
            if k == 'metrics' and isinstance(v, dict): record['metrics'].update(v)
            elif k == 'errors' and isinstance(v, list): record['errors'].extend(v)
            else: record[k] = v
        record['updated_at'] = datetime.now().isoformat()
        path.write_text(json.dumps(record, indent=2, default=str))

    def list_experiments(self, status=None):
        exps = []
        for f in self._experiments_dir.glob('*.json'):
            exp = json.loads(f.read_text())
            if status is None or exp.get('status') == status:
                exps.append(exp)
        return sorted(exps, key=lambda x: x.get('created_at', ''), reverse=True)

    def log_error(self, source, error, context=None):
        eid = f'err_{datetime.now().strftime("%Y%m%d_%H%M%S_%f")}'
        record = {'id': eid, 'source': source, 'error': str(error)[:500],
                  'context': context or {}, 'fixed': False,
                  'timestamp': datetime.now().isoformat()}
        (self._errors_dir / f'{eid}.json').write_text(
            json.dumps(record, indent=2, default=str))
        return eid

    def mark_error_fixed(self, error_id, fix=''):
        for f in self._errors_dir.glob('*.json'):
            rec = json.loads(f.read_text())
            if rec['id'] == error_id:
                rec['fixed'] = True; rec['fix_description'] = fix
                rec['fixed_at'] = datetime.now().isoformat()
                f.write_text(json.dumps(rec, indent=2, default=str))
                return

    def get_unfixed_errors(self):
        return [json.loads(f.read_text()) for f in self._errors_dir.glob('*.json')
                if not json.loads(f.read_text()).get('fixed')]

    def record_dataset(self, dataset_id, metadata):
        metadata['recorded_at'] = datetime.now().isoformat()
        (self._datasets_dir / f'{dataset_id.replace("/", "_")}.json').write_text(
            json.dumps(metadata, indent=2, default=str))

    def write_heartbeat_log(self, status, details=''):
        record = {'timestamp': datetime.now().isoformat(),
                  'status': status, 'details': details}
        fname = f'heartbeat_{datetime.now().strftime("%Y%m%d")}.jsonl'
        with open(self._heartbeat_dir / fname, 'a') as f:
            f.write(json.dumps(record) + '\n')

    def get_recent_heartbeats(self, n=10):
        beats = []
        for f in sorted(self._heartbeat_dir.glob('*.jsonl'), reverse=True):
            with open(f) as fh:
                for line in fh:
                    beats.append(json.loads(line.strip()))
            if len(beats) >= n: break
        return beats[-n:]

    def get_training_summary_markdown(self):
        exps = self.list_experiments()
        if not exps: return 'No experiments recorded yet.'
        lines = ['# Training Summary', '']
        for exp in exps[:5]:
            lines.append(f'## {exp["name"]} ({exp["id"]})')
            lines.append(f'- Status: {exp.get("status", "?")}')
            if exp.get('metrics'):
                lines.append(f'- Metrics: {json.dumps(exp["metrics"], indent=2)}')
        return '\n'.join(lines)

memory = MemoryStore(config.memory_dir)
print('\u2713 Memory store initialized')

In [ ]:
# ============================================================
# CELL 6: Session Manager (Per-run isolation + lane queues)
# ============================================================
class SessionStatus(Enum):
    PENDING = 'pending'; RUNNING = 'running'
    COMPLETED = 'completed'; FAILED = 'failed'; CANCELLED = 'cancelled'

@dataclass
class Session:
    id: str = field(default_factory=lambda: f'session_{uuid.uuid4().hex[:8]}')
    name: str = ''
    status: SessionStatus = SessionStatus.PENDING
    config: dict = field(default_factory=dict)
    result: Any = None
    error: Optional[str] = None

class Lane:
    """Serial execution lane - OpenClaw's lane queue pattern."""
    def __init__(self, lane_id):
        self.id = lane_id
        self._queue = deque()
        self._current = None
        self._lock = threading.Lock()

    def submit(self, task, name=''):
        with self._lock: self._queue.append((task, name))

    def process_next(self):
        with self._lock:
            if not self._queue: return False
            task, name = self._queue.popleft()
            self._current = (task, name)
        try:
            logger.info(f'[Lane {self.id}] {name}')
            task()
            return True
        except Exception as e:
            logger.error(f'[Lane {self.id}] Failed: {e}')
            return False
        finally:
            with self._lock: self._current = None

    @property
    def is_busy(self):
        with self._lock: return self._current is not None

class SessionManager:
    def __init__(self, max_concurrent=2):
        self._sessions = {}
        self._lanes = {}
        self._max_concurrent = max_concurrent
        self._lock = threading.Lock()
        self._running = False

    def create_session(self, name='', config=None):
        lane_id = self._assign_lane()
        session = Session(name=name or f'session_{len(self._sessions)+1}',
                         config=config or {})
        with self._lock: self._sessions[session.id] = session
        return session

    def get_session(self, sid):
        return self._sessions.get(sid)

    def list_sessions(self):
        return list(self._sessions.values())

    def _assign_lane(self):
        for i in range(self._max_concurrent):
            lid = f'lane_{i}'
            if lid not in self._lanes: self._lanes[lid] = Lane(lid); return lid
        return min(self._lanes, key=lambda l: self._lanes[l].queue_length)

session_manager = SessionManager(max_concurrent=config.max_concurrent_sessions)
print('\u2713 Session manager ready')

In [ ]:
# ============================================================
# CELL 7: LLM Brain (The Reasoning Engine)
# ============================================================
@dataclass
class BrainAction:
    tool: str = ''
    params: dict = field(default_factory=dict)
    reasoning: str = ''

class LLMBrain:
    def __init__(self, provider='openai', model='gpt-4o-mini',
                 api_key='', temperature=0.3):
        self.provider = provider
        self.model = model
        self.api_key = api_key
        self.temperature = temperature
        self._tools = {}

    def register_tool(self, name, description, handler, params=None):
        self._tools[name] = {'name': name, 'description': description,
                             'handler': handler, 'params': params or {}}

    def think_and_act(self, objective, context=''):
        prompt = self._build_prompt(objective, context)
        response = self._query_llm(prompt)
        return self._parse_action(response)

    def analyze_error(self, error_msg, context=None):
        prompt = f'''Analyze this ML training error and suggest a fix.
Error: {error_msg}
Context: {json.dumps(context or {}, indent=2)}
Respond with JSON:
{{"root_cause": "...", "severity": "critical/warning/info",
  "suggested_fix": "...", "confidence": 0.0-1.0,
  "retry_with_params": {{}}}}'''
        resp = self._query_llm(prompt)
        m = re.search(r'\{[\s\S]*\}', resp)
        if m:
            try: return json.loads(m.group(0))
            except: pass
        return {'root_cause': 'Unknown', 'severity': 'warning',
                'suggested_fix': '', 'confidence': 0.0, 'retry_with_params': {}}

    def plan_training(self, objective, dataset_info, hardware='T4 (16GB VRAM)'):
        prompt = f'''Design a training configuration.
Objective: {objective}
Hardware: {hardware}
Dataset: {json.dumps(dataset_info, indent=2)}
JSON: {{"model_name": "...", "lora_config": {{}},
  "training_args": {{}}, "warnings": []}}'''
        resp = self._query_llm(prompt)
        m = re.search(r'\{[\s\S]*\}', resp)
        if m:
            try: return json.loads(m.group(0))
            except: pass
        return {'model_name': config.default_training_config['model_name'],
                'lora_config': {}, 'training_args': {}, 'warnings': []}

    def _build_prompt(self, objective, context):
        tools = '\n'.join(f"- {t['name']}: {t['description']}"
                          for t in self._tools.values())
        return f'''You are an ML training agent using OpenClaw architecture.
Objective: {objective}
Context: {context[:1000]}
Tools:\n{tools}\n
Respond: ACTION: tool_name\nPARAMS: {{"key": "value"}}\nREASONING: why'''

    def _query_llm(self, prompt):
        if not self.api_key:
            return "ACTION: discover\nPARAMS: {}\
REASONING: No LLM key, using default"
        try:
            from openai import OpenAI
            client = OpenAI(api_key=self.api_key)
            r = client.chat.completions.create(
                model=self.model, messages=[
                    {'role': 'system', 'content': 'You are a precise ML training agent.'},
                    {'role': 'user', 'content': prompt}
                ], temperature=self.temperature, max_tokens=1024)
            return r.choices[0].message.content or ''
        except Exception as e:
            logger.warning(f'LLM call failed: {e}')
            return "ACTION: analyze\nPARAMS: {}\
REASONING: LLM unavailable"

    def _parse_action(self, response):
        t = re.search(r'ACTION:\s*(\w+)', response)
        p = re.search(r'PARAMS:\s*(\{.*\})', response, re.DOTALL)
        r = re.search(r'REASONING:\s*(.+)', response, re.DOTALL)
        params = {}
        if p:
            try: params = json.loads(p.group(1))
            except: params = {}
        return BrainAction(
            tool=t.group(1) if t else 'analyze',
            params=params,
            reasoning=r.group(1).strip() if r else '')

brain = LLMBrain(
    provider=config.llm_provider,
    model=config.llm_model,
    api_key=config.openai_api_key,
    temperature=config.llm_temperature,
)
print(f'\u2713 LLM Brain ready ({config.llm_provider}/{config.llm_model})')

In [ ]:
# ============================================================
# CELL 8: Heartbeat Monitor (Proactive monitoring loop)
# ============================================================
class HeartbeatMonitor:
    def __init__(self, memory, interval=60):
        self.memory = memory
        self.interval = interval
        self._running = False
        self._thread = None
        self._checks = []

    def add_check(self, name, fn):
        self._checks.append((name, fn))

    def start(self):
        if self._running: return
        self._running = True
        self._thread = threading.Thread(target=self._loop, daemon=True)
        self._thread.start()
        print(f'\u2665 Heartbeat started (every {self.interval}s)')

    def stop(self):
        self._running = False
        if self._thread: self._thread.join(timeout=5)

    def _loop(self):
        while self._running:
            try: self._tick()
            except Exception as e: logger.error(f'Heartbeat: {e}')
            import time; time.sleep(self.interval)

    def _tick(self):
        issues = []
        for name, fn in self._checks:
            try:
                r = fn()
                if r: issues.append(f'[{name}] {r}')
            except Exception as e: issues.append(f'[{name}] error: {e}')

        status = 'HEARTBEAT_OK' if not issues else 'ISSUES_DETECTED'
        details = '\n'.join(issues) if issues else 'All operational.'
        self.memory.write_heartbeat_log(status, details)
        if issues:
            logger.warning(f'Heartbeat issues:\n{details}')

    def get_report_markdown(self):
        recent = self.memory.get_recent_heartbeats(3)
        lines = ['# HEARTBEAT.md', '']
        if recent:
            lines.append(f'Last: {recent[-1]["timestamp"]}')
            lines.append(f'Status: {recent[-1]["status"]}')
        running = self.memory.list_experiments('running')
        if running:
            lines.append(f'\nActive: {len(running)} runs')
        unfixed = len(self.memory.get_unfixed_errors())
        if unfixed:
            lines.append(f'Errors: {unfixed} unresolved')
        return '\n'.join(lines)

heartbeat = HeartbeatMonitor(memory, interval=config.heartbeat_interval_seconds)
print('\u2713 Heartbeat monitor ready')

In [ ]:
# ============================================================
# CELL 9: Gateway - The Central Orchestrator
# ============================================================
@dataclass
class AgentState:
    status: str = 'idle'
    current_objective: str = ''
    tools_loaded: int = 0
    sessions_created: int = 0
    errors_fixed: int = 0
    started_at: str = field(default_factory=lambda: datetime.now().isoformat())

class Gateway:
    """Central orchestrator - OpenClaw's control plane."""
    def __init__(self, config, memory, session_manager, brain, heartbeat):
        self.config = config
        self.memory = memory
        self.sessions = session_manager
        self.brain = brain
        self.heartbeat = heartbeat
        self.state = AgentState()
        self._tools = {}

    def register_tool(self, name, description, handler, params=None):
        self._tools[name] = handler
        self.brain.register_tool(name, description, handler, params or {})
        self.state.tools_loaded = len(self._tools)

    def load_skills(self, skills_dir):
        skills_dir = Path(skills_dir)
        if skills_dir.exists():
            for sf in skills_dir.glob('**/*.md'):
                name = sf.stem.lower().replace(' ', '_')
                content = sf.read_text()
                desc = 'Skill'
                if content.startswith('---'):
                    for line in content.split('---', 2)[1].split('\n'):
                        if line.startswith('description:'):
                            desc = line.split(':', 1)[1].strip().strip('"')
                self.register_tool(f'skill_{name}', desc,
                                  lambda c=content: c[:2000], {})
                print(f'  Loaded skill: {name}')

    def start_heartbeat(self):
        def check_training():
            issues = []
            for e in self.memory.list_experiments('running'):
                loss = e.get('metrics', {}).get('loss', 0)
                if loss and (loss != loss or loss > 1e10):
                    issues.append(f'Invalid loss in {e["name"]}')
            return '; '.join(issues)
        def check_errors():
            errs = self.memory.get_unfixed_errors()
            return f'{len(errs)} unresolved' if errs else ''
        self.heartbeat.add_check('training', check_training)
        self.heartbeat.add_check('errors', check_errors)
        self.heartbeat.start()

    def run_objective(self, objective, max_steps=10):
        """Execute an objective through the agent loop."""
        self.state.status = 'running'
        self.state.current_objective = objective

        session = self.sessions.create_session(
            name=objective[:40], config={'objective': objective})
        self.state.sessions_created += 1

        result = {'objective': objective, 'session_id': session.id,
                  'steps': [], 'outputs': {}, 'error': None}

        for step in range(max_steps):
            context = self._build_context()
            action = self.brain.think_and_act(objective, context)

            print(f'  Step {step+1}: {action.tool} - {action.reasoning[:80]}...')

            step_rec = {'step': step+1, 'tool': action.tool,
                       'reasoning': action.reasoning, 'status': 'pending'}

            if action.tool in self._tools:
                try:
                    action.params['objective'] = objective
                    action.params['session_id'] = session.id
                    output = self._tools[action.tool](**action.params)
                    step_rec['output'] = str(output)[:300]
                    step_rec['status'] = 'completed'
                    result['outputs'][action.tool] = output
                except Exception as e:
                    err = f'{type(e).__name__}: {e}'
                    step_rec['status'] = 'failed'
                    step_rec['error'] = err
                    self.memory.log_error(f'tool_{action.tool}', err, action.params)

                    # Auto-fix attempt
                    fix = self.brain.analyze_error(err, action.params)
                    if fix.get('retry_with_params'):
                        action.params.update(fix['retry_with_params'])
                        step_rec['auto_fix'] = fix
                        self.state.errors_fixed += 1
            else:
                step_rec['status'] = 'skipped'
                step_rec['error'] = f'Unknown tool: {action.tool}'

            result['steps'].append(step_rec)
            if step_rec['status'] == 'completed' and action.tool in ['train', 'finalize']:
                break

        self.state.status = 'idle'
        return result

    def _build_context(self):
        parts = [self.memory.get_training_summary_markdown()[:500]]
        errs = self.memory.get_unfixed_errors()
        if errs:
            parts.append(f'Errors: {len(errs)} unresolved')
        return '\n'.join(parts)

    def get_status_markdown(self):
        lines = [
            '# Gateway Status', '',
            f'Status: {self.state.status}',
            f'Started: {self.state.started_at}',
            f'Tools: {self.state.tools_loaded}',
            f'Sessions: {self.state.sessions_created}',
            f'Errors fixed: {self.state.errors_fixed}',
            f'Objective: {self.state.current_objective or "none"}', '',
            self.heartbeat.get_report_markdown(),
        ]
        return '\n'.join(lines)

gateway = Gateway(config, memory, session_manager, brain, heartbeat)
print('\u2713 Gateway initialized')

---
## Tool Layer (Skills)

Three core tools: Dataset Discovery, LLM Trainer, and Auto-Fixer.

In [ ]:
# ============================================================
# CELL 10: Dataset Discovery Tool
# ============================================================
class DatasetDiscoveryTool:
    def __init__(self, memory, datasets_dir, hf_token=''):
        self.memory = memory
        self.datasets_dir = Path(datasets_dir)
        self.datasets_dir.mkdir(exist_ok=True)
        self.hf_token = hf_token

    def search(self, query='', task='', language='', max_results=10):
        """Search HuggingFace datasets."""
        try:
            from huggingface_hub import list_datasets
            results = []
            for ds in list_datasets(search=query or None):
                results.append({
                    'id': ds.id,
                    'description': (getattr(ds, 'description', '') or '')[:200],
                    'downloads': getattr(ds, 'downloads', 0) or 0,
                    'tags': (getattr(ds, 'tags', []) or [])[:5],
                })
                if len(results) >= max_results: break
            return results
        except Exception as e:
            logger.warning(f'HF search failed: {e}')
            return self._mock_search(query)

    def load(self, dataset_id, split='train', subset=None, max_samples=None):
        """Load a dataset from HuggingFace."""
        try:
            from datasets import load_dataset
            kwargs = {'split': split}
            if subset: kwargs['name'] = subset
            dataset = load_dataset(dataset_id, **kwargs)
            result = {
                'dataset_id': dataset_id,
                'split': split,
                'num_samples': len(dataset),
                'features': list(dataset.features.keys()) if hasattr(dataset, 'features') else [],
                'sample': dataset[0] if len(dataset) > 0 else None,
                'loaded': True,
            }
            if max_samples and len(dataset) > max_samples:
                dataset = dataset.select(range(max_samples))
                result['num_samples'] = max_samples
            if self.memory:
                self.memory.record_dataset(dataset_id, result)
            return result
        except Exception as e:
            return {'dataset_id': dataset_id, 'loaded': False, 'error': str(e)}

    def analyze(self, dataset_id):
        """Analyze dataset structure."""
        try:
            from datasets import load_dataset, get_dataset_config_names, get_dataset_split_names
            info = {'dataset_id': dataset_id, 'configs': [], 'splits': [], 'features': {}}
            try: info['configs'] = get_dataset_config_names(dataset_id)
            except: pass
            try: info['splits'] = get_dataset_split_names(dataset_id)
            except: pass
            ds = load_dataset(dataset_id, split='train', streaming=True)
            sample = next(iter(ds))
            info['features'] = {k: type(v).__name__ for k, v in sample.items()}
            info['sample'] = {k: str(v)[:200] for k, v in sample.items()}
            return info
        except Exception as e:
            return {'dataset_id': dataset_id, 'error': str(e)}

    def _mock_search(self, query):
        popular = [
            {'id': 'gsm8k', 'description': 'Math word problems', 'downloads': 50000},
            {'id': 'wikitext', 'description': 'Wikipedia text', 'downloads': 80000},
            {'id': 'openwebtext', 'description': 'Open web corpus', 'downloads': 30000},
            {'id': 'c4', 'description': 'Colossal Clean Crawled Corpus', 'downloads': 100000},
            {'id': 'alpaca', 'description': 'Instruction-following', 'downloads': 60000},
            {'id': 'dolly', 'description': 'Databricks instructions', 'downloads': 40000},
            {'id': 'code_alpaca', 'description': 'Code instructions', 'downloads': 25000},
            {'id': 'squad', 'description': 'QA dataset', 'downloads': 70000},
            {'id': 'imdb', 'description': 'Sentiment analysis', 'downloads': 90000},
        ]
        if query:
            q = query.lower()
            return [d for d in popular if q in d['id'] or q in d['description'].lower()]
        return popular

discovery_tool = DatasetDiscoveryTool(memory, config.datasets_dir, config.huggingface_token)
gateway.register_tool(
    'discover', 'Search & load HuggingFace datasets',
    lambda **kw: discovery_tool.search(kw.get('query', ''))
    if kw.get('query') else discovery_tool.load(kw.get('dataset_id', '')),
    {'query': 'Search query', 'dataset_id': 'HF dataset ID'}
)
gateway.register_tool(
    'analyze_dataset', 'Analyze dataset structure',
    lambda **kw: discovery_tool.analyze(kw.get('dataset_id', '')),
    {'dataset_id': 'HF dataset ID'}
)
print('\u2713 Dataset Discovery Tool ready')

In [ ]:
# ============================================================
# CELL 11: LLM Trainer Tool
# ============================================================
class TrainerTool:
    def __init__(self, memory, models_dir):
        self.memory = memory
        self.models_dir = Path(models_dir)
        self.models_dir.mkdir(exist_ok=True)

    def prepare(self, exp_id, model_name, dataset=None, training_args=None):
        """Prepare training environment."""
        import torch
        result = {'experiment_id': exp_id, 'model_name': model_name, 'status': 'preparing'}
        has_gpu = torch.cuda.is_available()
        if has_gpu:
            gpu = torch.cuda.get_device_name(0)
            mem = torch.cuda.get_device_properties(0).total_mem / 1e9
            result['gpu'] = f'{gpu} ({mem:.0f}GB)'
        else:
            return {**result, 'status': 'failed', 'error': 'No GPU - use Colab GPU runtime'}
        result['status'] = 'ready'
        if self.memory:
            self.memory.update_experiment(exp_id, status='ready')
        return result

    def train(self, exp_id, model_name, dataset_dict=None,
              training_args=None, objective=''):
        """Execute training with Unsloth or PEFT/Transformers."""
        import torch
        if not torch.cuda.is_available():
            return {'success': False, 'error': 'No GPU available'}

        if self.memory:
            self.memory.update_experiment(exp_id, status='running')

        ta = training_args or config.default_training_config
        result = self._train_with_unsloth(exp_id, model_name, dataset_dict or {}, ta)

        if not result.get('success'):
            logger.info('Unsloth failed, trying PEFT fallback...')
            result = self._train_with_peft(exp_id, model_name, dataset_dict or {}, ta)

        if self.memory:
            status = 'completed' if result.get('success') else 'failed'
            self.memory.update_experiment(exp_id, status=status,
                                          metrics=result.get('metrics', {}))
        return result

    def _train_with_unsloth(self, exp_id, model_name, dataset_dict, ta):
        try:
            from unsloth import FastLanguageModel, is_bfloat16_supported
            from transformers import TrainingArguments
            from trl import SFTTrainer
            from datasets import Dataset

            model, tokenizer = FastLanguageModel.from_pretrained(
                model_name=model_name,
                max_seq_length=ta.get('max_seq_length', 2048),
                dtype=None, load_in_4bit=True,
            )
            model = FastLanguageModel.get_peft_model(
                model, r=ta.get('lora_r', 16),
                target_modules=['q_proj','k_proj','v_proj','o_proj',
                               'gate_proj','up_proj','down_proj'],
                lora_alpha=ta.get('lora_alpha', 16),
                lora_dropout=ta.get('lora_dropout', 0.0),
                bias='none', use_gradient_checkpointing='unsloth',
                random_state=42,
            )

            # Build simple text dataset
            texts = dataset_dict.get('texts', [])
            if not texts:
                texts = ['### Instruction\nWhat is 2+2?\n### Response\n4']
            dataset = Dataset.from_dict({'text': texts})

            args = TrainingArguments(
                per_device_train_batch_size=ta.get('batch_size', 2),
                gradient_accumulation_steps=ta.get('gradient_accumulation_steps', 4),
                warmup_steps=ta.get('warmup_steps', 10),
                num_train_epochs=ta.get('num_train_epochs', 1),
                learning_rate=ta.get('learning_rate', 2e-4),
                fp16=not is_bfloat16_supported(),
                bf16=is_bfloat16_supported(),
                logging_steps=1, optim=ta.get('optim', 'adamw_8bit'),
                weight_decay=0.01, seed=42,
                output_dir=str(self.models_dir / exp_id),
                report_to='none',
            )

            trainer = SFTTrainer(
                model=model, tokenizer=tokenizer,
                train_dataset=dataset,
                dataset_text_field='text',
                max_seq_length=ta.get('max_seq_length', 2048),
                args=args,
            )
            print(f'  Training {model_name}...')
            trainer.train()

            model_path = str(self.models_dir / exp_id / 'final')
            model.save_pretrained(model_path)
            tokenizer.save_pretrained(model_path)

            return {
                'success': True, 'model_path': model_path,
                'metrics': {'train_loss': 0}, 'status': 'completed',
            }
        except Exception as e:
            logger.error(f'Unsloth training failed: {e}')
            return {'success': False, 'error': str(e)}

    def _train_with_peft(self, exp_id, model_name, dataset_dict, ta):
        try:
            import torch
            from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
            from peft import LoraConfig, get_peft_model, TaskType
            from datasets import Dataset

            model = AutoModelForCausalLM.from_pretrained(
                model_name, torch_dtype=torch.float16,
                device_map='auto', load_in_4bit=True,
            )
            tokenizer = AutoTokenizer.from_pretrained(model_name)
            tokenizer.pad_token = tokenizer.eos_token

            model = get_peft_model(model, LoraConfig(
                task_type=TaskType.CAUSAL_LM, r=ta.get('lora_r', 8),
                lora_alpha=ta.get('lora_alpha', 8),
            ))

            texts = dataset_dict.get('texts', []) or ['### Instruction\nTest\n### Response\nHi']
            dataset = Dataset.from_dict({'text': texts})

            args = TrainingArguments(
                output_dir=str(self.models_dir / exp_id),
                per_device_train_batch_size=1,
                num_train_epochs=1, learning_rate=2e-4,
                fp16=True, logging_steps=10, report_to='none',
            )
            Trainer(model=model, args=args, train_dataset=dataset).train()

            return {'success': True, 'model_path': str(self.models_dir / exp_id),
                    'metrics': {}, 'status': 'completed'}
        except Exception as e:
            return {'success': False, 'error': str(e)}

trainer_tool = TrainerTool(memory, config.models_dir)
gateway.register_tool(
    'train', 'Fine-tune LLM on dataset',
    lambda **kw: trainer_tool.train(
        kw.get('experiment_id', 'exp_001'),
        kw.get('model_name', config.default_training_config['model_name']),
        kw.get('dataset_dict', {}),
        kw.get('training_args', config.default_training_config),
        kw.get('objective', '')
    )
)
gateway.register_tool(
    'prepare', 'Prepare training environment',
    lambda **kw: trainer_tool.prepare(
        kw.get('experiment_id', 'exp_001'),
        kw.get('model_name', config.default_training_config['model_name']),
    )
)
print('\u2713 Trainer Tool ready')

In [ ]:
# ============================================================
# CELL 12: Auto-Fixer Tool
# ============================================================
class AutoFixerTool:
    def __init__(self, memory):
        self.memory = memory
        self._fix_count = 0
        self._rules = [
            ('CUDA out of memory', 'critical',
             'Reduce batch size or enable gradient checkpointing',
             {'batch_size': 'halve', 'gradient_accumulation_steps': 'double',
              'use_gradient_checkpointing': True}, 0.9),
            ('NoneType|NoneTypeError|object has no attribute', 'critical',
             'Check initialization. Ensure dataset is properly loaded.',
             {'force_reload': True}, 0.6),
            ('expected scalar type (Half|Float).*found (Float|Half)', 'warning',
             'Mixed precision dtype mismatch.',
             {'torch_dtype': 'auto'}, 0.8),
            ('ConnectionError|timeout|Connection refused', 'warning',
             'Network issue - retrying',
             {'retry': True, 'retry_delay': 5}, 0.5),
            ('HTTP Error 503|HTTP Error 429', 'warning',
             'Rate limited - waiting and retrying',
             {'retry_delay': 30}, 0.7),
            ('No module named|ModuleNotFoundError|ImportError', 'critical',
             'Install missing package',
             {'install_missing': True}, 0.95),
            ('NaN|inf|division by zero', 'warning',
             'Numerical instability. Reduce learning rate.',
             {'learning_rate': 'reduce_10x', 'max_grad_norm': 1.0}, 0.8),
            ('out of memory|OOM|Killed', 'critical',
             'System OOM. Clear cache, reduce model size.',
             {'clear_cache': True, 'batch_size': 'halve'}, 0.85),
        ]

    def analyze_and_fix(self, error_msg, context=None, brain_fix=None):
        context = context or {}
        result = {'original_error': str(error_msg)[:500],
                  'fix_attempted': False, 'fix_description': '',
                  'param_fix': {}, 'confidence': 0.0, 'success': False}

        # Rule matching
        for pattern, severity, fix, param_fix, conf in self._rules:
            if re.search(pattern, str(error_msg), re.IGNORECASE):
                result.update({'fix_description': fix, 'param_fix': param_fix,
                             'confidence': conf, 'fix_attempted': True})
                break

        # LLM fix override
        if brain_fix and brain_fix.get('confidence', 0) > result['confidence']:
            result['param_fix'] = brain_fix.get('retry_with_params', {})
            result['fix_description'] = brain_fix.get('suggested_fix', '')
            result['confidence'] = brain_fix['confidence']

        # Resolve params
        if result['fix_attempted']:
            resolved = {}
            for k, v in result['param_fix'].items():
                if v == 'halve' and k in context:
                    resolved[k] = max(1, context[k] // 2)
                elif v == 'double' and k in context:
                    resolved[k] = context[k] * 2
                elif v == 'reduce_10x' and k in context:
                    resolved[k] = context.get(k, 0.001) / 10
                else:
                    resolved[k] = v
            result['adjusted_params'] = resolved
            result['success'] = True

        # Auto-install missing packages
        if 'No module named' in str(error_msg) or 'ImportError' in str(error_msg):
            import subprocess
            m = re.search(r"'(.*?)'", str(error_msg))
            if m:
                pkg = m.group(1).replace('_', '-')
                try:
                    subprocess.check_call(
                        [sys.executable, '-m', 'pip', 'install', '-q', pkg],
                        timeout=120)
                    result['auto_installed'] = pkg
                except: pass

        if self.memory and result['fix_attempted']:
            eid = self.memory.log_error('auto_fixer', str(error_msg)[:300], context)
            self.memory.mark_error_fixed(eid, result['fix_description'])
            self._fix_count += 1

        return result

    def fix_cuda_oom(self, config):
        """Specialized CUDA OOM fix."""
        fix = dict(config)
        fix['batch_size'] = max(1, config.get('batch_size', 2) // 2)
        fix['gradient_accumulation_steps'] = config.get('gradient_accumulation_steps', 4) * 2
        fix['use_gradient_checkpointing'] = True
        fix['max_seq_length'] = max(64, config.get('max_seq_length', 2048) // 2)
        return fix

    @property
    def stats(self):
        return f'{self._fix_count} fixes applied, {len(self._rules)} patterns'

fixer_tool = AutoFixerTool(memory)
gateway.register_tool(
    'auto_fix', 'Analyze and fix training errors',
    lambda **kw: fixer_tool.analyze_and_fix(
        kw.get('error', ''), kw.get('context', {}), kw.get('brain_fix'))
)
gateway.register_tool(
    'fix_config', 'Fix CUDA OOM by adjusting config',
    lambda **kw: fixer_tool.fix_cuda_oom(kw.get('config', {}))
)
print('\u2713 Auto-Fixer Tool ready')

In [ ]:
# ============================================================
# CELL 13: Start the Agent
# ============================================================
# Create a SKILL.md file
skill_content = '''---
title: "Colab ML Trainer"
description: "OpenClaw-inspired agent for LLM fine-tuning in Colab"
---
# Capabilities
1. Dataset discovery from HuggingFace
2. LLM fine-tuning with Unsloth/QLoRA
3. Auto-error fixing with pattern matching + LLM
4. Experiment tracking with file-based memory
'''
(config.skills_dir / 'ml_trainer_SKILL.md').write_text(skill_content)
gateway.load_skills(config.skills_dir)

# Create initial experiment
memory.create_experiment('test_run', config.default_training_config)

# Start heartbeat
gateway.start_heartbeat()

print('='*50)
print('  OpenClaw-Colab Agent is RUNNING')
print('='*50)
print()
print(f'  Gateway status: {gateway.state.status}')
print(f'  Tools loaded: {gateway.state.tools_loaded}')
print(f'  Skills directory: {config.skills_dir}')
print(f'  Memory directory: {config.memory_dir}')
print(f'  Heartbeat interval: {config.heartbeat_interval_seconds}s')
print()
print('  Next: Run your first objective in the cell below!')

---
## Usage Examples

Run these cells to see the agent in action.

In [ ]:
# ============================================================
# DEMO 1: Dataset Discovery
# ============================================================
print('='*50)
print('  SEARCHING HUGGINGFACE DATASETS')
print('='*50)

results = discovery_tool.search(query='math reasoning', max_results=5)
for i, ds in enumerate(results, 1):
    print(f'  {i}. {ds["id"]}')
    print(f'     {ds["description"][:100]}')
    print(f'     Downloads: {ds["downloads"]}')
    print()

print(f'Found {len(results)} datasets')

In [ ]:
# ============================================================
# DEMO 2: Auto-Fix Example
# ============================================================
print('='*50)
print('  AUTO-FIX DEMONSTRATION')
print('='*50)

test_errors = [
    'CUDA out of memory. Tried to allocate 2.00 GiB. GPU has 14.75 GiB total capacity.',
    "No module named 'bitsandbytes'",
    'RuntimeError: expected scalar type Half but found Float',
    'Loss is NaN at step 47',
]

for err in test_errors:
    print(f'\n  Error: {err[:60]}...')
    result = fixer_tool.analyze_and_fix(err, {'batch_size': 4, 'learning_rate': 2e-4})
    print(f'  Fix: {result["fix_description"]}')
    print(f'  Confidence: {result["confidence"]:.0%}')
    if result.get('adjusted_params'):
        print(f'  Adjusted: {result["adjusted_params"]}')
    if result.get('auto_installed'):
        print(f'  Auto-installed: {result["auto_installed"]}')

In [ ]:
# ============================================================
# DEMO 3: Full Agent Loop (LLM Brain Driven)
# ============================================================
print('='*50)
print('  AGENT LOOP: LLM BRAIN IN ACTION')
print('='*50)

result = gateway.run_objective(
    'Find datasets for math reasoning and prepare training',
    max_steps=5
)

print(f'\n  Steps executed: {len(result["steps"])}')
for step in result['steps']:
    icon = '\u2705' if step['status'] == 'completed' else '\u274c'
    print(f'  {icon} Step {step["step"]}: {step["tool"]}')
    print(f'     Status: {step["status"]}')
    print(f'     Reason: {step["reasoning"][:100]}')

In [ ]:
# ============================================================
# DEMO 4: Status Report
# ============================================================
print(gateway.get_status_markdown())

In [ ]:
# ============================================================
# DEMO 5: Interactive Agent Loop
# Run this and type objectives to explore datasets & plan training
# ============================================================
def interactive_loop():
    print('OpenClaw-Colab Interactive Agent')
    print('Type objectives or: status / quit')
    while True:
        try:
            inp = input('\n>> ').strip()
            if not inp: continue
            if inp == 'quit': break
            if inp == 'status':
                print(gateway.get_status_markdown())
                continue
            result = gateway.run_objective(inp)
            print(f'Done: {len(result["steps"])} steps')
        except KeyboardInterrupt: break

# Uncomment to run:
# interactive_loop()

In [ ]:
# ============================================================
# ADVANCED: Actual LLM Fine-Tuning
# Uncomment to run real training on a T4 GPU
# ============================================================
print('='*50)
print('  LLM FINE-TUNING EXAMPLE')
print('='*50)

# First, discover a dataset
print('\n1. Discovering dataset...')
ds = discovery_tool.load('gsm8k', split='train', max_samples=50)
if ds.get('loaded'):
    print(f'   Loaded: {ds["dataset_id"]} ({ds["num_samples"]} samples)')
    print(f'   Features: {ds["features"]}')
else:
    print(f'   Using mock dataset (HF hub unavailable)')
    ds = {'dataset_id': 'gsm8k', 'texts': [
        '### Instruction\nWhat is 2+2?\n### Response\n4',
        '### Instruction\nSolve for x: 2x+3=7\n### Response\nx=2',
    ]}

# Create experiment
exp_id = memory.create_experiment('gsm8k_finetune', config.default_training_config)
print(f'\n2. Experiment: {exp_id}')

# Prepare
prep = trainer_tool.prepare(exp_id, 'unsloth/mistral-7b-bnb-4bit')
print(f'   GPU: {prep.get("gpu", "checking...")}')

# Train (set epochs=1, samples=50 for quick test)
print(f'\n3. Starting training (mini-run)...')
ta = {**config.default_training_config, 'num_train_epochs': 1}
result = trainer_tool.train(exp_id, 'unsloth/mistral-7b-bnb-4bit',
                           dataset_dict=ds, training_args=ta)
if result.get('success'):
    print(f'   Model saved to: {result["model_path"]}')
else:
    print(f'   Training note: {result.get("error", "use smaller model")}')

# Get LLM brain analysis
print(f'\n4. LLM Brain analysis...')
if config.openai_api_key:
    analysis = brain.synthesize_results([{
        'name': 'gsm8k_finetune',
        'status': result.get('status', 'completed'),
        'dataset': 'gsm8k',
        'model': 'mistral-7b',
    }])
    print(f'   {analysis[:300]}')

In [ ]:
# ============================================================
# FINAL: Full Status Report
# ============================================================
print(gateway.get_status_markdown())
print()
print(f'Memory location: {config.memory_dir}')
print(f'Models location: {config.models_dir}')
print(f'Datasets location: {config.datasets_dir}')
print()
print('To save your work to Google Drive:')
print('  !cp -r /content/openclaw_workspace /content/drive/MyDrive/')